<a href="https://colab.research.google.com/github/Bruno-Paulo/PETs/blob/main/Homomorphic_Encryption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homomorphic Encryption Tutorial

_Last run: May 1, 2025_


## 1. Introduction

In this tutorial we will explore Homomorphic Encryption (HE), a cryptographic technique that allows computations to be performed on encrypted data without having to decrypt it first. This ensures that privacy is maintained while still allowing meaningful analysis.

We will apply HE using the TenSEAL library to securely process medical data. At the end of this tutorial you will understand how to encrypt, process and decrypt data using homomorphic encryption while maintaining patient confidentiality.

### 1.1 Scenario

Imagine a healthcare analytics company (call it LifeMed) that builds predictive models to assess stroke risk based on patients' medical records. The company wants to collaborate with external researchers to refine its models and improve stroke prediction.

However, patient privacy regulations such as GDPR and HIPAA prohibit the sharing of raw medical data. If LifeMed provides unencrypted patient data, it could expose sensitive health information, violate compliance regulations and compromise patient confidentiality.

**How can LifeMed allow researchers to analyse the data without ever exposing it?**

### 1.2 Solution

Homomorphic encryption allows LifeMed to encrypt patient data before it is shared, allowing researchers to perform encrypted calculations without ever seeing the raw data. When their analysis is complete, LifeMed can decrypt only the final results, while all intermediate calculations remain hidden.

This approach ensures that:

- Patient data remains confidential throughout the analysis.
- Researchers can still perform meaningful computations, such as risk assessment or statistical analysis.
- LifeMed complies with strict privacy regulations.


### 1.3 Homomorphic Encryption

Homomorphic encryption (HE) is a type of encryption that allows computations to be performed on ciphertext (encrypted data) without decrypting it. The output remains encrypted and only the owner of the data can decrypt the final result.

There are three main types of HE:

- Partially Homomorphic Encryption (PHE): Supports only addition or multiplication (not both).

- Somewhat Homomorphic Encryption (SHE): Supports a limited number of addition and multiplication operations.

- Fully Homomorphic Encryption (FHE): Supports unlimited computations on encrypted data (the most powerful, but computationally expensive).

In this tutorial we will use FHE with CKKS encryption, a scheme optimised for privacy-preserving machine learning, which allows efficient computations on encrypted floating-point numbers.

### 1.4 Outline of this tutorial

By the end of this tutorial, you will have a solid understanding of homomorphic encryption using TenSEAL and how to apply it to secure medical data analysis.

#### **Setting up the encryption context**
We will configure a CKKS encryption context in TenSEAL and adjust parameters for security and performance.

#### **Medical data encryption and processing**
We will encrypt patient data and perform secure calculations, such as calculating the average age of stroke patients without revealing their actual age.

#### **Decrypting and interpreting results**
Finally, we will decrypt the processed data and validate if our encrypted calculations have retained their accuracy.


## 2. Setup

In order to perform this tutorial, we will need to download the original data and then configure the Python library that will assist in the homomorphic encryption process.


---


***Note:*** This code will only work correctly if you use the Google
Chrome browser.



---

### 2.1 Downloading the original dataset

Please download the healthcare dataset in CSV file format from [here](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset).

Now we need attach the CSVs to this Google Colab notebook. Run the code below and then Click the "Choose Files" button. In the file picker choose the CSV file that you have just downloaded.


In [ ]:
from google.colab import files

# Optional: You can skip this step if you are running the code on your own
# machine
uploaded = files.upload()

Saving healthcare-dataset-stroke-data.csv to healthcare-dataset-stroke-data.csv


The csv files are now available in the folder `content/`.

### 2.2 Installing the TenSEAL library

TenSEAL is a library for performing homomorphic encryption operations on tensors, based on [Microsoft SEAL](https://github.com/Microsoft/SEAL). It provides ease of use through a Python API, while maintaining efficiency by implementing most of its operations in C++. TenSEAL provides tools for:

- Encryption/Decryption of vectors of integers using BFV.
- Encryption/Decryption of vectors of real numbers using CKKS.
- Element-wise addition, subtraction and multiplication of encrypted/encrypted vectors and encrypted/plain vectors.
- Dot product and vector-matrix multiplication.

If you need more help [Browse all tutorials](https://github.com/OpenMined/TenSEAL?tab=readme-ov-file#tutorials).

To install the TenSEAL library, run the following:

In [ ]:
%pip install tenseal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 28.2 MB/s eta 0:00:00


## 3. Operations on Data



---



***Note:*** This introduction assumes basic familiarity with PyTorch, so it doesn't cover the PyTorch-related aspects in full detail. If you want to dive deeper into PyTorch, we recommend [*DEEP LEARNING WITH PYTORCH: A 60 MINUTE BLITZ*](https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html).



---

### 3.1 Pre-process the data

We will create the load_data function which will load and pre-process the data. This process is divided into smaller steps and functions.


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE

DEVICE = "cpu"
print(f"Training on {DEVICE}")
print(f"PyTorch {torch.__version__}")

Training on cpu
PyTorch 2.6.0+cu124


#### 3.1.1 Handle missing values and drop unnecessary columns

We clean the dataset by handling missing values and removing irrelevant columns.

In [ ]:
def clean_data(df):
    df.drop(columns=["id"], inplace=True)  # Drop irrelevant columns
    df["bmi"] = df["bmi"].fillna(df["bmi"].median())  # Fill missing BMI values
    return df

#### 3.1.2 Split features and target

We separate the input features (X) from the target variable (y).

In [ ]:
def split_features_target(df):
    X = df.drop(columns=["stroke"])
    y = df["stroke"]
    return X, y

#### 3.1.3 Preprocess categorical and numerical features

We define transformations for numerical and categorical columns.

In [ ]:
def preprocess_features(X, y):
    categorical_features = X.select_dtypes(include=["object"]).columns
    numeric_features = X.select_dtypes(include=["float64", "int64"]).columns

    preprocessor = ColumnTransformer(transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ])

    X_transformed = preprocessor.fit_transform(X)
    return np.array(X_transformed, dtype=np.float32), np.array(y, dtype=np.float32)

#### 3.1.4 Handle class imbalance with SMOTE

SMOTE is used to balance the dataset.

In [ ]:
def balance_classes(X, y):
    return SMOTE().fit_resample(X, y)

#### 3.1.5 Split data into train, validation, and test sets

We split the dataset into training, validation, and testing sets.

In [ ]:
def split_data(X, y, test_size=0.3, val_size=0.5):
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=test_size, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=val_size, random_state=42)
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

#### 3.1.6 Convert data to pyTorch datasets

We define a custom dataset class and create DataLoaders.

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def create_dataloaders(train, val, test, batch_size=32):
    train_dataset = CustomDataset(*train)
    val_dataset = CustomDataset(*val)
    test_dataset = CustomDataset(*test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

#### 3.1.7 Load and process data using these functions

Now we call all the above functions in sequence to load and process the data.

In [ ]:
def load_data():
    df = pd.read_csv('/content/healthcare-dataset-stroke-data.csv')
    df = clean_data(df)
    X, y = split_features_target(df)
    X, y = preprocess_features(X, y)
    X, y = balance_classes(X, y)
    train, val, test = split_data(X, y)
    return create_dataloaders(train, val, test)

### 3.2 Creating a logistic regression model

We will use this model as a means of comparison against encrypted training and evaluation.

#### 3.2.1 Stroke prediction model class

In [ ]:
# Define Logistic Regression model
class LR(nn.Module):
    def __init__(self, n_features):
        super(LR, self).__init__()
        self.lr = nn.Linear(n_features, 1)  # Output size 1 for binary classification

    def forward(self, x):
        return torch.sigmoid(self.lr(x))  # Sigmoid activation for binary classification

#### 3.2.2 Train function

In [ ]:
# Training function
def train(model, optimizer, criterion, train_loader, epochs=5):
    model.train()  # Set model to training mode
    for epoch in range(1, epochs + 1):
        total_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            batch_y = batch_y.view(-1, 1)  # Reshape to match model output shape

            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch}/{epochs}, Loss: {avg_loss:.4f}")

#### 3.2.3 Accuracy function

In [ ]:
def accuracy(model, data_loader):
    model.eval()  # Set model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in data_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            batch_y = batch_y.view(-1, 1)  # Reshape to match model output

            output = model(batch_x)
            predicted = (output >= 0.5).float()  # Convert probabilities to binary predictions
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    return correct / total  # Return accuracy as a fraction

#### 3.2.4 Training the model

In [ ]:
# Load data
train_loader, val_loader, test_loader = load_data()

# Get number of features from dataset
n_features = next(iter(train_loader))[0].shape[1]  # Extract feature size from first batch

# Initialize model, loss function, and optimizer
model = LR(n_features).to(DEVICE)
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=1.0)

# Train model
train(model, optimizer, criterion, train_loader, epochs=5)

# Calculate accuracy on test set
plain_accuracy = accuracy(model, test_loader)
print(f"Accuracy on test set: {plain_accuracy:.4f}")

Epoch 1/5, Loss: 0.4808
Epoch 2/5, Loss: 0.4765
Epoch 3/5, Loss: 0.4758
Epoch 4/5, Loss: 0.4767
Epoch 5/5, Loss: 0.4776
Accuracy on test set: 0.7875


### 3.3 Creating an encrypted logistic regression model

#### 3.3.1 Encrypted model

In [ ]:
class EncryptedLR:
    def __init__(self, model):
    #  with torch.no_grad():  # Ensure no gradient tracking
          self.weights = model.lr.weight.squeeze(0).tolist()  # Convert tensor to list
          self.bias = model.lr.bias.item()  # Convert bias tensor to scalar

    def forward(self, enc_X):
        # We don't need to perform sigmoid as this model
        # will only be used for evaluation, and the label
        # can be deduced without applying sigmoid
        enc_out = enc_X.dot(self.weights) + self.bias
        return enc_out

    ################################################
    ## You can use the functions below to perform ##
    ## the evaluation with an encrypted model     ##
    ################################################

    def encrypt(self, context):
        self.weights = ts.ckks_vector(context, self.weights)
        self.bias = ts.ckks_vector(context, [self.bias])

    def decrypt(self):
        self.weights = self.weights.decrypt()
        self.bias = self.bias.decrypt()[0]  # Extract scalar from decrypted vector

#### 3.3.2 Context

The **TenSEALContext** is a special object that manages encryption keys and parameters, simplifying encrypted calculations by encapsulating all the necessary details. Instead of handling keys separately, users create a **single** TenSEALContext to perform encrypted operations.




In [ ]:
import tenseal as ts

# Create TenSEAL encryption context
def create_ckks_context():
    poly_mod_degree = 8192
    coeff_mod_bit_sizes = [40, 21, 21, 21, 21, 21, 21, 40]

    ctx = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_mod_degree,
        coeff_mod_bit_sizes
    )

    ctx.global_scale = 2 ** 21
    ctx.generate_galois_keys()
    return ctx

---

**Encryption scheme**

TenSEAL supports two encryption algorithms:

- BFV (Brakerski/Fan-Vercauteren): Ideal for exact calculations on integers.
- CKKS (Cheon-Kim-Kim-Song): Supports approximate arithmetic, mainly used for encrypted calculations on floating point numbers.


**Polynomial modulus degree** (`poly_modulus_degree`)

The polynomial modulus degree determines the size of the polynomials used in encryption, must be a power of 2, and has a direct effect on:

- The number of coefficients in plaintext polynomials.
- The size of the ciphertext elements (higher values increase the size of the ciphertext).
- Computational performance (higher degrees slow down computations).
- Security level (higher degrees provide stronger security).

<!--- **plain_modulus:** The plaintext modulus. Should not be passed if the scheme is CKKS. -->

**Coefficient modulus bit sizes** (`coeff_mod_bit_sizes`)

The coefficient modulus consists of a list of prime numbers used for encryption. These primes have an effect:

- The size of the ciphertext elements (larger values increase the size).
- The multiplication depth (more primes allow more encrypted multiplications).
- The level of security (larger values reduce security).
- Reducing the coefficient modulus length reduces the context size, but also reduces the available multiplication depth.
- Smaller coefficient modulus sizes reduce context size but lower precision (especially for CKKS).

Each prime in the coefficient modulus must be at most 60 bits and congruent to 1 mod (2 × `poly_modulus_degree`).


---

#### 3.3.3 Encryption

The next step after creating our TenSEALContext is to start doing some encrypted computation. First, we create an encrypted vector.

In [ ]:
# Encrypt a single feature vector
def encrypt_vector(context, feature_vector):
    return ts.ckks_vector(context, feature_vector.tolist())

#### 3.3.4 Additional TenSEALContext attributes

In addition to encryption parameters, TenSEALContext manages several useful attributes:

- **Automatic relinearisation** (`auto_relin`): Simplifies the reduction of ciphertext size after multiplications.

- **Automatic rescaling** (`auto_rescale`) (CKKS only): Automatically adjusts scaling factors to maintain precision.

- **Automatic modulus switching** (`auto_mod_switch`): Reduces noise by adjusting the modulus during calculations.

These features are enabled by default, but can be manually controlled for advanced use cases.

In [ ]:
# create context
context = create_ckks_context()

print("Automatic relinearization is:", ("on" if context.auto_relin else "off"))
print("Automatic rescaling is:", ("on" if context.auto_rescale else "off"))
print("Automatic modulus switching is:", ("on" if context.auto_mod_switch else "off"))

Automatic relinearization is: on
Automatic rescaling is: on
Automatic modulus switching is: on


### 3.4 Encrypted evaluation

Let's create a Client/Server framework for encrypted machine learning using homomorphic encryption, where the server provide services, such as model evaluation over encrypted inputs, then the client can encrypt his model input and send it to the server for evaluation and get back the encrypted result.

#### 3.4.1 Client

The client, LifeMed, will prepare the data, encrypt it, and send it to the server, external researchers.

Then, after the encrypted evaluation done by the server (described ahead), the client will decrypt the result.

##### 3.4.1.1 Prepare and encrypt query

In [ ]:
# Client-side: Prepare and encrypt a query
def prepare_encrypted_query(context, sample):
    enc_sample = encrypt_vector(context, sample)
    server_context = context.copy()
    server_context.make_context_public()  # Remove secret key for security

    return {
        "data": enc_sample.serialize(),
        "context": server_context.serialize()
    }

##### 3.4.1.2 Decrypt result

In [ ]:
# Client-side: Decrypt result
def decrypt_response(context, response):
    enc_result = ts.ckks_vector_from(context, response["data"])
    decrypted_value = enc_result.decrypt()[0]  # Extract single prediction

    # Convert to probability using sigmoid function
    probability = 1 / (1 + np.exp(-decrypted_value))

    # Determine stroke prediction
    prediction = "Stroke" if probability > 0.5 else "No Stroke"

    return probability, prediction

##### 3.4.1.3 Accuracy

In [ ]:
def encrypted_accuracy(decrypted_results, y_test):
    """Computes accuracy based on decrypted predictions."""
    correct = 0
    total = len(y_test)

    for (probability, _), true_label in zip(decrypted_results, y_test):
        # Convert probability to binary prediction
        prediction = 1 if probability > 0.5 else 0

        # Compare with true label
        if prediction == int(true_label):
            correct += 1

    accuracy = correct / total
    print(f"Encrypted Model Accuracy: {correct}/{total} = {accuracy:.4f}")

    return accuracy

#### 3.4.2 Server

The server, the external researchers, will decrypt the context, process the encrypted query and return the encrypted result.

In [ ]:
def approximate_sigmoid(enc_x):
    """Polynomial approximation of sigmoid function for encrypted values."""
    return 0.5 + (0.197 * enc_x) #- (0.004 * (enc_x * enc_x * enc_x))  # x^3

def server_process_request(model, query, enc_true_label):
    server_ctx = ts.context_from(query["context"])
    enc_sample = ts.ckks_vector_from(server_ctx, query["data"])

    enc_prediction = model.forward(enc_sample)  # Encrypted output

    # Use polynomial approximation for sigmoid
    sigmoid_enc_prediction = approximate_sigmoid(enc_prediction)

    # Compute encrypted loss using BCE approximation (no log)
    enc_loss = -(enc_true_label * sigmoid_enc_prediction +
                 (1 - enc_true_label) * (1 - sigmoid_enc_prediction))

    return {"data": enc_prediction.serialize(), "loss": enc_loss.serialize()}


#### 3.4.3 Running the workflow

Let's run the implemented framework and see the results.





##### 3.4.3.1 Load test data

We will start by defining the number of samples for our workflow.


---


***Note:*** HE is CPU-intensive and very time consuming, so for this demo we advise you to use a maximum of 100 samples from our dataset, and even that will take around 10 minutes to execute. If you want to execute it faster, please use only 10 or 50 samples.



---

In [ ]:
# Define number of samples
SAMPLES = 100

In [ ]:
def load_test_data():
    """Loads test data and returns all test samples as tensors."""
    train_loader, val_loader, test_loader = load_data()
    all_test_samples = list(test_loader)  # Convert DataLoader to a list
    X_all = torch.cat([batch[0] for batch in all_test_samples])  # Stack features
    y_all = torch.cat([batch[1] for batch in all_test_samples])  # Stack labels
    return X_all, y_all

##### 3.4.3.2 Select random samples

In [ ]:
def select_random_samples(X_all, y_all, num_samples=SAMPLES):
    """Selects a random subset of samples for encryption."""
    subset_indices = torch.randperm(len(X_all))[:num_samples]  # Shuffle and pick indices
    X_subset = X_all[subset_indices].numpy()  # Convert to NumPy
    y_subset = y_all[subset_indices].numpy()  # Convert to NumPy
    return X_subset, y_subset

##### 3.4.3.3 Encryption

In [ ]:
def encrypt_samples(context, X_subset):
    """Encrypts all selected samples."""
    return [prepare_encrypted_query(context, X_sample) for X_sample in X_subset]


def process_encrypted_requests(model, encrypted_queries, encrypted_labels):
    """Processes encrypted queries on the server with corresponding encrypted labels."""
    return [server_process_request(model, query, enc_label)
            for query, enc_label in zip(encrypted_queries, encrypted_labels)]

##### 3.4.3.4 Decryption

In [ ]:
def decrypt_results(context, encrypted_responses):
    """Decrypts and interprets the results."""
    return [decrypt_response(context, response) for response in encrypted_responses]

##### 3.4.3.5 Sumarise results

In [ ]:
def summarise_results(decrypted_results):
    """Prints and summarises the results."""
    print(f"Summary Statistics for the first 10 Samples:")
    for i, (probability, prediction) in enumerate(decrypted_results[:10]):
        print(f"Sample {i+1}: Probability = {probability:.4f}, Prediction = {prediction}")

    # Convert results to a NumPy array for analysis
    probabilities, predictions = zip(*decrypted_results)
    probabilities = np.array(probabilities)
    predictions = np.array([1 if pred == "Stroke" else 0 for pred in predictions])

    # Output summary statistics
    print(f"\nSummary Statistics for {SAMPLES} Samples:")
    print(f"Average Stroke Probability: {probabilities.mean():.4f}")
    print(f"Stroke Cases Predicted: {predictions.sum()} / {len(predictions)}")

##### 3.4.3.6 Execution

In [ ]:
X_all, y_all = load_test_data()
X_subset, y_subset = select_random_samples(X_all, y_all)

# Initialize encryption context
context = create_ckks_context()

# Encrypt the selected samples
encrypted_queries = encrypt_samples(context, X_subset)

# Load the encrypted model on the server
encrypted_model = EncryptedLR(model)

# Encrypt the weights
encrypted_model.encrypt(context)

encrypted_labels = [ts.ckks_vector(context, [label]) for label in y_subset]  # Encrypt each label
encrypted_responses = process_encrypted_requests(encrypted_model, encrypted_queries, encrypted_labels)

decrypted_results = decrypt_results(context, encrypted_responses)
summarise_results(decrypted_results)

# Compute encrypted model accuracy
encrypted_model_accuracy = encrypted_accuracy(decrypted_results, y_subset)

print(f"Final Encrypted Model Accuracy: {encrypted_model_accuracy:.4f}")
diff_accuracy = plain_accuracy - encrypted_model_accuracy
print(f"Difference between plain and encrypted accuracies: {diff_accuracy}")
if diff_accuracy < 0:
    print("Oh! We got a better accuracy on the encrypted test-set! The noise was on our side...")

Summary Statistics for the first 10 Samples:
Sample 1: Probability = 0.6812, Prediction = Stroke
Sample 2: Probability = 0.1944, Prediction = No Stroke
Sample 3: Probability = 0.2128, Prediction = No Stroke
Sample 4: Probability = 0.7075, Prediction = Stroke
Sample 5: Probability = 0.3593, Prediction = No Stroke
Sample 6: Probability = 0.4792, Prediction = No Stroke
Sample 7: Probability = 0.1796, Prediction = No Stroke
Sample 8: Probability = 0.1518, Prediction = No Stroke
Sample 9: Probability = 0.6240, Prediction = Stroke
Sample 10: Probability = 0.5148, Prediction = Stroke

Summary Statistics for 100 Samples:
Average Stroke Probability: 0.4207
Stroke Cases Predicted: 41 / 100
Encrypted Model Accuracy: 56/100 = 0.5600
Final Encrypted Model Accuracy: 0.5600
Difference between plain and encrypted accuracies: 0.22752570253598348


The reduced accuracy of the encrypted model is primarily due to the limitations of homomorphic encryption.

- HE does not natively support non-linear operations.

- The more complex the model in terms of number of layers, the more computational overhead. Simple models should be used instead.

- The CKKS scheme introduces approximation errors that affect logistic regression calculations, particularly the sigmoid function.

- Since non-linear functions cannot be computed directly, we approximate the sigmoid with polynomials, resulting in deviations in the classification probabilities.

- Encrypted operations consume a fixed noise budget, leading to numerical instability over multiple computations.

- Accuracy is expected to vary across different subsets due to randomness in the sampling.

## 4. Challenge: Average age of stroke patients

Your task is to use TenSEAL to calculate the average age of stroke patients without decoding the individual ages. For this exercise use the following parameters:

- `poly_modulus_degree = 8192` - adjust for security/performance trade-off.
- `coeff_mod_bit_sizes = [60, 40, 40, 60]` - controls precision and noise budget.
- `context.global_scale = 2**40` - adjust for precision.


**Steps to complete the challenge**

1. Load and preprocess the dataset
2. Extract the "age" column and filter it to include only patients who have had a stroke.
3. Encrypt the ages using TenSEAL context with CKKS.
4. Compute the average age while keeping the data encrypted (homomorphic computation).
5. Decrypt the result and check that it is matches the plaintext computation.

**Expected result**

A decrypted average age that matches what you would get using plaintext computation.

The solutions can be found at the end of the tutorial.


**Instructions to submit the code**

1. Write your code on the code block below and test it out.
2. Save it as txt file.
3. Submit it on the [Homomorphic Encryption forms](https://forms.gle/pjYRJLZj8xhv9zN68).

In [ ]:
# First Write and test your code here



---


***Note:***
(Extra) - Try and experiment also with the initial TenSEAL parameters defined in the begining of the tutorial and check the differences in the predicted result. Why does it happen?

---


## 5. Conclusions

In this tutorial, we explored **homomomorphic encryption (HE)** using the **TenSEAL** library to securely process medical data while preserving patient privacy. We implemented a logistic regression model, encrypted patient records, and performed encrypted inference using the **CKKS** scheme. By simulating a **client-server** framework, we demonstrated how external researchers can analyse encrypted data without ever accessing raw patient information. Finally, we discussed the trade-offs of HE, such as computational overhead and reduced accuracy due to function approximation.


## Solutions

### Solution for average age of stroke patients




#### 1. Load and Preprocess the dataset

In [ ]:
# Load dataset (modify this line based on your actual dataset)
df = pd.read_csv('/content/healthcare-dataset-stroke-data.csv')

# Filter for stroke patients
stroke_ages = df[df["stroke"] == 1]["age"].values  # Extract ages of stroke patients

print(f"Total Stroke Patients: {len(stroke_ages)}")
print(f"Plaintext Average Age: {np.mean(stroke_ages):.2f}")


Total Stroke Patients: 249
Plaintext Average Age: 67.73


#### 2. Encryption context

In [ ]:
def create_ckks_context():
    """Creates and configures a CKKS TenSEAL context."""
    context = ts.context(
        scheme=ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,  # Adjust for security/performance trade-off
        coeff_mod_bit_sizes=[60, 40, 40, 60]  # Controls precision and noise budget
    )
    context.global_scale = 2**40  # Adjust for precision
    context.generate_galois_keys()  # Needed for vector operations
    return context

context = create_ckks_context()

#### 3. Encrypt ages

In [ ]:
def encrypt_ages(context, ages):
    """Encrypts the age values using CKKS."""
    return ts.ckks_vector(context, ages.tolist())

enc_ages = encrypt_ages(context, stroke_ages)
print("Ages encrypted successfully!")

Ages encrypted successfully!


#### 4. Compute encrypted average age



---
***Note:*** Fully homomorphic encryption does not support division operations. Instead we multiply by the inverse.


---




In [ ]:
def compute_encrypted_average(enc_ages, num_patients):
    """Computes the encrypted average age using homomorphic operations."""
    encrypted_sum = enc_ages.sum()  # Homomorphic sum of encrypted ages
    inverse_num_patients = 1 / num_patients  # Inverse of the number of patients
    encrypted_average = encrypted_sum * inverse_num_patients  # Homomorphic multiplication
    return encrypted_average

enc_avg_age = compute_encrypted_average(enc_ages, len(stroke_ages))
print("Encrypted average age computed successfully!")

Encrypted average age computed successfully!


#### 5. Decrypt the result

In [ ]:
def decrypt_result(context, enc_avg_age):
    """Decrypts the encrypted average age and returns the result."""
    return enc_avg_age.decrypt()[0]  # Extracts the single value from CKKS vector

decrypted_avg_age = decrypt_result(context, enc_avg_age)
print(f"Decrypted Average Age: {decrypted_avg_age:.2f}")
print(f"Plaintext Check: {np.mean(stroke_ages):.2f}")

Decrypted Average Age: 67.73
Plaintext Check: 67.73


#### Change encryption settings

The context defined in the begining of the tutorial uses smaller bit sizes, leading to more rounding errors. Also, that context has a much smaler scale, leading to more precision loss compared to the context created for the exercise.

---
***Note:*** Re-run the `create_ckks_context()` function defined in the begining of the tutorial and then re-run steps 3-5 from the solution.

Try modifying the encryption settings:
- Change `poly_modulus_degree`: increase for better security, but worse performance.

- Adjust `global_scale`: higher values improve precision but increase ciphertext size.

- Reduce `coeff_mod_bit_sizes`: reduces noise budget, impacting computation depth.

---



